In [ ]:
!nvidia-smi

Sun Mar 22 19:49:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install unsloth


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 130.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
from google.colab import files
uploaded = files.upload()
print(uploaded.keys())

Saving tdata.json to tdata.json
dict_keys(['tdata.json'])


In [ ]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'unsloth_compiled_cache', 'tdata.json', 'sample_data']


In [ ]:
import json
from datasets import Dataset
from unsloth import FastLanguageModel

# Load your raw examples
with open("tdata.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Load model + tokenizer first
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=4096,
    load_in_4bit=True,
)

training_texts = []

for example in data:
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    training_texts.append(text)

dataset = Dataset.from_dict({"text": training_texts})

print("Total examples:", len(dataset))

==((====))==  Unsloth 2026.3.10: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Total examples: 100


In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

model = FastLanguageModel.get_peft_model(
    model,
    r=16, #rank, the higer the more learning / weight of adapter / increase as data size grows
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32, # scalling factor = r * 2
    lora_dropout=0, # deterministic
    bias="none", # only apply to weights
    use_gradient_checkpointing="unsloth", #use unsloth to redduce vram usage
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    dataset_num_proc=2,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4, #effective batch size is 2*4 =8
        warmup_steps=5,
        num_train_epochs=3,#
        learning_rate=2e-4,#
        fp16=True,#
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,#
        lr_scheduler_type="linear",#
        seed=3407,#
        output_dir="outputs_qwen3_4b",
        report_to="none",
    ),
)

trainer.train()

Unsloth 2026.3.10 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 3 | Total steps = 39
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.587606
2,2.635951
3,2.319761
4,2.214780
5,1.882336
6,1.642418
7,1.617081
8,1.547235
9,1.360218
10,1.300441


TrainOutput(global_step=39, training_loss=1.0838068097065656, metrics={'train_runtime': 551.7086, 'train_samples_per_second': 0.544, 'train_steps_per_second': 0.071, 'total_flos': 5696926422220800.0, 'train_loss': 1.0838068097065656, 'epoch': 3.0})

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "system",
        "content": "You are an expert interview coach. Evaluate the candidate strictly using the STAR method: Situation, Task, Action, Result. Use the intake metadata to tailor your evaluation to the target role. Return concise, structured feedback. Score performance from 1 to 10 based primarily on STAR quality, clarity, relevance, ownership, and measurable impact. Return valid JSON only with keys: grade, summary, star, mockAnswer, suggestions."
    },
    {
        "role": "user",
        "content": """{
  "intake": {
    "jobType": "internship",
    "lastName": "Singh",
    "position": "Software Engineering Intern",
    "firstName": "Sehaj",
    "companyName": "Google",
    "jobDescription": "Software engineering internship focused on building scalable products and internal tools. Strong candidates show programming fundamentals, data structures and algorithms knowledge, debugging ability, teamwork, and clear communication. Experience from coursework, projects, internships, or open-source work is valued."
  },
  "transcript": [
    {
      "role": "interviewer",
      "content": "Tell me about a time you had to solve a difficult technical problem while working with others."
    },
    {
      "role": "candidate",
      "content": "In one of my software projects for school, my team was building a recurring payment reminder app. Near the end of development, reminders were being sent at the wrong times for some users, especially when payment dates were edited. It was a big problem because the main purpose of the app was accurate reminders, and the bug was affecting trust in the product. I took responsibility for investigating the scheduling logic and fixing the issue before our demo. First I reproduced the bug using different due date scenarios and found that the logic was too tightly coupled to the API flow. I proposed separating the date calculation and recurrence logic into a dedicated service so we could test it independently. I then wrote test cases for monthly and custom intervals, updated the reminder workflow, and worked with my teammate to verify the fix across multiple user cases. After the changes, reminders matched expected dates in our test cases and we were able to demo the app successfully. The project taught me the importance of isolating business logic, testing edge cases early, and communicating clearly with teammates when debugging under time pressure."
    }
  ]
}"""
    }
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')

outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True, temperature=0.7, do_sample=True, top_p=0.9)

response = tokenizer.batch_decode(outputs)[0]

print(response)

Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


RuntimeError: output with shape [1, 32, 1, 128] doesn't match the broadcast shape [1, 32, 478, 128]

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "system",
        "content": "You are an expert interview coach. Evaluate the candidate strictly using the STAR method: Situation, Task, Action, Result. Use the intake metadata to tailor your evaluation to the target role. Return concise, structured feedback. Score performance from 1 to 10 based primarily on STAR quality, clarity, relevance, ownership, and measurable impact. Return valid JSON only with keys: grade, summary, star, mockAnswer, suggestions."
    },
    {
        "role": "user",
        "content": """{
  "intake": {
    "jobType": "internship",
    "lastName": "Singh",
    "position": "Software Engineering Intern",
    "firstName": "Sehaj",
    "companyName": "Google",
    "jobDescription": "Software engineering internship focused on building scalable products and internal tools. Strong candidates show programming fundamentals, data structures and algorithms knowledge, debugging ability, teamwork, and clear communication. Experience from coursework, projects, internships, or open-source work is valued."
  },
  "transcript": [
    {
      "role": "interviewer",
      "content": "Tell me about a time you had to solve a difficult technical problem while working with others."
    },
    {
      "role": "candidate",
      "content": "In one of my software projects for school, my team was building a recurring payment reminder app. Near the end of development, reminders were being sent at the wrong times for some users, especially when payment dates were edited. It was a big problem because the main purpose of the app was accurate reminders, and the bug was affecting trust in the product. I took responsibility for investigating the scheduling logic and fixing the issue before our demo. First I reproduced the bug using different due date scenarios and found that the logic was too tightly coupled to the API flow. I proposed separating the date calculation and recurrence logic into a dedicated service so we could test it independently. I then wrote test cases for monthly and custom intervals, updated the reminder workflow, and worked with my teammate to verify the fix across multiple user cases. After the changes, reminders matched expected dates in our test cases and we were able to demo the app successfully. The project taught me the importance of isolating business logic, testing edge cases early, and communicating clearly with teammates when debugging under time pressure."
    }
  ]
}"""
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)

new_tokens = outputs[0][inputs.shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(response)

Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RuntimeError: output with shape [1, 32, 1, 128] doesn't match the broadcast shape [1, 32, 478, 128]

In [ ]:
from unsloth import FastLanguageModel
import torch

FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "system",
        "content": "You are an expert interview coach. Evaluate the candidate strictly using the STAR method: Situation, Task, Action, Result. Use the intake metadata to tailor your evaluation to the target role. Return concise, structured feedback. Score performance from 1 to 10 based primarily on STAR quality, clarity, relevance, ownership, and measurable impact. Return valid JSON only with keys: grade, summary, star, mockAnswer, suggestions."
    },
    {
        "role": "user",
        "content": """{
  "intake": {
    "jobType": "internship",
    "lastName": "Singh",
    "position": "Software Engineering Intern",
    "firstName": "Sehaj",
    "companyName": "Google",
    "jobDescription": "Software engineering internship focused on building scalable products and internal tools. Strong candidates show programming fundamentals, data structures and algorithms knowledge, debugging ability, teamwork, and clear communication. Experience from coursework, projects, internships, or open-source work is valued."
  },
  "transcript": [
    {
      "role": "interviewer",
      "content": "Tell me about a time you had to solve a difficult technical problem while working with others."
    },
    {
      "role": "candidate",
      "content": "In one of my software projects for school, my team was building a recurring payment reminder app. Near the end of development, reminders were being sent at the wrong times for some users, especially when payment dates were edited. It was a big problem because the main purpose of the app was accurate reminders, and the bug was affecting trust in the product. I took responsibility for investigating the scheduling logic and fixing the issue before our demo. First I reproduced the bug using different due date scenarios and found that the logic was too tightly coupled to the API flow. I proposed separating the date calculation and recurrence logic into a dedicated service so we could test it independently. I then wrote test cases for monthly and custom intervals, updated the reminder workflow, and worked with my teammate to verify the fix across multiple user cases. After the changes, reminders matched expected dates in our test cases and we were able to demo the app successfully. The project taught me the importance of isolating business logic, testing edge cases early, and communicating clearly with teammates when debugging under time pressure."
    }
  ]
}"""
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt",
    truncation=True,
    max_length=4096,
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RuntimeError: output with shape [1, 32, 1, 128] doesn't match the broadcast shape [1, 32, 478, 128]

In [ ]:
model.save_pretrained_gguf('finetuned_model', tokenizer, quantization_method='q4_k_m')

Unsloth: Merging model weights to 16-bit format...


config.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [04:16<00:00, 128.30s/it]


Unsloth: Merge process complete. Saved to `/content/finetuned_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['finetuned_model_gguf/qwen3-4b-instruct-2507.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['finetuned_model_gguf/qwen3-4b-instruct-2507.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model finetuned_model_gguf/qwen3-4b-instruct-2507.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to finetuned_model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f finetuned_model_gguf/Modelfile


{'save_directory': 'finetuned_model',
 'gguf_directory': 'finetuned_model_gguf',
 'gguf_files': ['finetuned_model_gguf/qwen3-4b-instruct-2507.Q4_K_M.gguf'],
 'modelfile_location': 'finetuned_model_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
!zip -r finetuned_model_gguf.zip finetuned_model_gguf

  adding: finetuned_model_gguf/ (stored 0%)
  adding: finetuned_model_gguf/qwen3-4b-instruct-2507.Q4_K_M.gguf (deflated 2%)
  adding: finetuned_model_gguf/Modelfile (deflated 59%)


In [ ]:
from google.colab import files
files.download("finetuned_model_gguf.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/finetuned_model_gguf /content/drive/MyDrive/

In [ ]:
!cp -r /content/finetuned_model /content/drive/MyDrive/

In [ ]:
!ls -lah /content/drive/MyDrive/finetuned_model_gguf
!ls -lah /content/drive/MyDrive/finetuned_model

total 2.4G
-rw------- 1 root root 1.6K Mar 22 23:33 Modelfile
-rw------- 1 root root 2.4G Mar 22 23:33 qwen3-4b-instruct-2507.Q4_K_M.gguf
total 7.6G
drwx------ 3 root root 4.0K Mar 22 23:34 .cache
-rw------- 1 root root 4.0K Mar 22 23:34 chat_template.jinja
-rw------- 1 root root 1.8K Mar 22 23:34 config.json
-rw------- 1 root root 4.7G Mar 22 23:36 model-00001-of-00002.safetensors
-rw------- 1 root root 2.9G Mar 22 23:37 model-00002-of-00002.safetensors
-rw------- 1 root root  33K Mar 22 23:34 model.safetensors.index.json
-rw------- 1 root root 4.5K Mar 22 23:34 tokenizer_config.json
-rw------- 1 root root  11M Mar 22 23:34 tokenizer.json


In [ ]:
!pip install vllm

In [ ]:
!vllm serve /content/finetuned_model \
  --trust-remote-code \
  --port 8000 \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.80 \
  --enforce-eager

2026-03-22 23:54:16.856293: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774223656.881934   69934 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774223656.889553   69934 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774223656.907628   69934 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774223656.907660   69934 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774223656.907664   69934 computation_placer.cc:177] computation placer alr

In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/v1/chat/completions",
    json={
        "model": "/content/finetuned_model",
        "messages": [
            {"role": "system", "content": "You are an expert interview coach. Return valid JSON only."},
            {"role": "user", "content": "Evaluate this interview response: I led a team project where we fixed a scheduling bug by isolating business logic and adding tests."}
        ],
        "temperature": 0.2,
        "max_tokens": 300
    },
    timeout=120,
)

print(response.status_code)
print(response.text)

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fab1d490410>: Failed to establish a new connection: [Errno 111] Connection refused'))